# Fase 3 · Núcleo algorítmico, eficiencia y POO
## Licitaciones públicas de ChileCompra · MCDI500 · Grupo 2

F3 integra las reglas de limpieza y transformación desarrolladas en F2 dentro de una arquitectura modular y orientada a objetos (`src/preprocesamiento.py` + `src/pipeline_licitaciones.py`), contrasta sus dos salidas con las exportaciones de F2 mediante igualdad exacta y evalúa el rendimiento y la complejidad algorítmica de sus decisiones clave. La unidad de análisis de la brecha es la licitación, mientras que la base de ofertas conserva el detalle por oferta e ítem.

**Integración del avance formativo.** La entrega formativa no se presentó en un repositorio o cuaderno separado: sus scripts de medición temporal/espacial y el análisis de complejidad asintótica y empírica se integraron íntegramente dentro de esta Fase 3, en `src/medicion.py` y la sección 4.

**Cómo ejecutar.** Deja el archivo de entrada en `data/raw/04LicitacionesEne.csv` (ver *Obtener los datos* en el `README.md`), ejecuta primero `F2/F2_Preprocesamiento.ipynb` para generar las dos referencias en `data/processed/` y luego ejecuta todas las celdas de este notebook con el kernel **Python (grupo2-mcdi500)**. Los módulos compartidos están en `src/`: `preprocesamiento.py`, `pruebas_preprocesamiento.py`, `pipeline_licitaciones.py`, `pruebas_pipeline.py` y `medicion.py`.


## 0. Verificar el entorno de ejecución

Antes de cargar datos comprobamos que el notebook se ejecuta con el entorno virtual del proyecto (`.venv`, kernel **Python (grupo2-mcdi500)**) y que las librerías que usa esta fase tienen las versiones declaradas en `requirements.txt`. Es la misma verificación de la sección 8 de `F1_Definicion`, reducida a lo que necesita la Fase 3.

Importa porque las mediciones de tiempo y memoria de las secciones siguientes dependen del entorno: si cambia la versión de pandas, pueden cambiar las cifras. La celda se detiene si no hay entorno virtual o si falta una librería, y avisa si alguna versión no coincide con la declarada.

In [1]:
# Verificar el entorno: intérprete, entorno virtual y versiones frente a requirements.txt.
import platform
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

# Paquetes que usa la Fase 3, con el nombre que tienen en requirements.txt.
PAQUETES_F3 = ["pandas", "numpy"]


def leer_versiones_declaradas(ruta_requirements, paquetes):
    """Devuelve {paquete: versión} de requirements.txt para los paquetes pedidos."""
    declaradas = {}
    for linea in ruta_requirements.read_text(encoding="utf-8").splitlines():
        nombre, _, version = linea.strip().partition("==")
        if nombre.lower() in paquetes:
            declaradas[nombre.lower()] = version
    return declaradas


def verificar_entorno(paquetes):
    """Comprueba el entorno virtual y compara versiones instaladas y declaradas.

    Se detiene si no hay entorno virtual o si falta un paquete. Devuelve una
    tabla con la versión declarada e instalada de cada paquete.
    """
    inicio = Path.cwd().resolve()
    raiz = next((p for p in (inicio, *inicio.parents) if (p / "requirements.txt").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró requirements.txt: abre el notebook dentro del repositorio.")
    if sys.prefix == sys.base_prefix:
        raise RuntimeError("No se usa un entorno virtual: cambia el kernel a 'Python (grupo2-mcdi500)'.")

    declaradas = leer_versiones_declaradas(raiz / "requirements.txt", paquetes)
    filas = []
    for paquete in paquetes:
        try:
            instalada = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            instalada = None
        filas.append({"Paquete": paquete, "Declarada": declaradas.get(paquete),
                      "Instalada": instalada, "Coincide": instalada == declaradas.get(paquete)})
    tabla = pd.DataFrame(filas)

    faltantes = tabla.loc[tabla["Instalada"].isna(), "Paquete"].tolist()
    if faltantes:
        raise ModuleNotFoundError(f"Faltan paquetes: {faltantes}. Ejecuta: pip install -r requirements.txt")

    interprete = Path(sys.executable)
    if interprete.is_relative_to(raiz):
        interprete = interprete.relative_to(raiz)
    print(f"Intérprete: {interprete} | Python {platform.python_version()} | {platform.system()}")
    print("[OK] Entorno virtual activo")
    return tabla


versiones = verificar_entorno(PAQUETES_F3)
display(versiones)
if not versiones["Coincide"].all():
    print("[AVISO] Hay versiones distintas a las declaradas: las cifras de tiempo pueden variar.")
else:
    print(f"[OK] Las {len(versiones)} versiones coinciden con requirements.txt")

Intérprete: .venv\Scripts\python.exe | Python 3.15.2 | Windows
[OK] Entorno virtual activo


,Paquete,Declarada,Instalada,Coincide
0,pandas,3.1.0,3.1.0,True
1,numpy,2.5.4,2.5.4,True


[OK] Las 2 versiones coinciden con requirements.txt


## 1. Configurar el proyecto y ejecutar las pruebas de F2

In [2]:
from io import StringIO

inicio = Path.cwd()
raiz = next((p for p in (inicio, *inicio.parents) if (p / "src/pipeline_licitaciones.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("No se encontró src/pipeline_licitaciones.py: abre el notebook dentro del repositorio.")
sys.path.insert(0, str(raiz))

from src.preprocesamiento import FECHAS, MONTOS, MOTIVOS, ORDEN_TAMANO, cargar_csv, es_columna_privada
from src.pipeline_licitaciones import (
    Bitacora, ExtremosLogIQR, TratamientoFaltantes, construir_pipeline,
    control_iterativo, control_vectorizado,
)
from src.pruebas_preprocesamiento import ejecutar_pruebas_preprocesamiento

entrada = raiz / "data/raw/04LicitacionesEne.csv"
destino = raiz / "data/processed"
referencias = {
    "ofertas": destino / "04LicitacionesEne_procesado.csv",
    "licitaciones": destino / "04LicitacionesEne_brecha_por_licitacion.csv",
}
for ruta in [entrada, *referencias.values()]:
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta {ruta.relative_to(raiz)}. Verifica la entrada y ejecuta F2 primero.")
ejecutar_pruebas_preprocesamiento()
print("Pruebas de las reglas de F2 aprobadas.")

Pruebas de las reglas de F2 aprobadas.


## 2. Integración en F3: pipeline ajustable y principios de diseño

El pipeline aplica en todas sus etapas el contrato `ajustar(datos)` → `transformar(datos)`. En `src/pipeline_licitaciones.py`, `Pipeline.ejecutar()` recorre la secuencia construida por `construir_pipeline()` y llama ambos métodos en cada paso; no se usa una API alternativa como `fit_transform` ni se omite el ajuste de las reglas fijas.

- **Abstracción y herencia.** `Transformador(ABC)` define el contrato común y el método abstracto `_aplicar()`. `TransformacionFija`, `TratamientoFaltantes` y `ExtremosLogIQR` heredan de esa base e implementan su regla concreta en `src/pipeline_licitaciones.py`.
- **Encapsulamiento y validación.** En `Transformador.ajustar()` y `Transformador.transformar()` se validan `columnas_requeridas`, se trabaja con copias defensivas y se controla `_ajustado`; `transformar()` detiene el flujo si no hubo ajuste previo. `TratamientoFaltantes` conserva sus columnas aprendidas en `_columnas_excluidas`, y `ExtremosLogIQR.limites` entrega una copia de `_limites`.
- **Template Method.** `Transformador.ajustar()` y `Transformador.transformar()` fijan el orden de validación, copia y control de estado y delegan el trabajo específico a `_ajustar_en()` y `_aplicar()` de cada subclase.
- **Adaptador.** `TransformacionFija` envuelve las funciones de `src/preprocesamiento.py` (`marcar_duplicados`, `convertir_tipos` y `ordenar_tamano`) para que también cumplan `ajustar()` → `transformar()`.
- **Strategy y polimorfismo.** `construir_pipeline()` combina adaptadores y transformadores con estado en una sola secuencia. `Pipeline.ejecutar()` aplica el mismo bucle a todos, incluidos `TratamientoFaltantes` (aprende columnas con más de 90 % de faltantes) y `ExtremosLogIQR` (aprende límites logarítmicos para CLP).
- **División funcional.** Las reglas de datos se mantienen en `src/preprocesamiento.py`; el contrato y la coordinación orientada a objetos están en `src/pipeline_licitaciones.py`.

Las siguientes comprobaciones ejercitan el ajuste y la transformación de las estrategias con estado. Después, `Pipeline.ejecutar(entrada)` ejecuta de forma uniforme las cinco etapas del pipeline real sobre el CSV de entrada (el diseño y la evidencia del patrón **Observer** con `Bitacora` se desarrollan en la sección 8).

In [3]:
from src.preprocesamiento import ordenar_tamano
from src.pipeline_licitaciones import TransformacionFija

# Una regla determinista de F2 se adapta al mismo contrato ajustable de F3.
base_ordinal = pd.DataFrame({"TamanoProveedor": ["Micro", "Mediana"]})
adaptador_ordinal = TransformacionFija(
    "Ordenar tamaño de proveedor", ordenar_tamano, ("TamanoProveedor",)
)
salida_ordinal = adaptador_ordinal.ajustar(base_ordinal).transformar(base_ordinal)
assert salida_ordinal["TamanoProveedorOrdinal"].cat.ordered
assert "TamanoProveedorOrdinal" not in base_ordinal
print("Se comprobó TransformacionFija.ajustar() → transformar() sin mutar la entrada.")

ajuste = pd.DataFrame({"MonedaOferta": ["CLP"] * 5, "MontoTotalOferta": [10, 20, 30, 40, 50]})
evaluacion = pd.DataFrame({"MonedaOferta": ["CLP", "USD"], "MontoTotalOferta": [1000, 1000]})
detector = ExtremosLogIQR()
detector.ajustar(ajuste)
bandera = detector.transformar(evaluacion)["EsExtremoLogIQR_CLP"]
pd.testing.assert_series_equal(
    bandera, pd.Series([True, pd.NA], dtype="boolean", name="EsExtremoLogIQR_CLP")
)
print("Se comprobó que ExtremosLogIQR reutiliza los límites aprendidos.")

ajuste_falt = pd.DataFrame({"A": [1, 2, 3], "Vacia90": [pd.NA, pd.NA, pd.NA]})
eval_falt = pd.DataFrame({"A": [4], "Vacia90": ["dato"]})
trat_falt = TratamientoFaltantes()
trat_falt.ajustar(ajuste_falt)
assert trat_falt.columnas_excluidas == ("Vacia90",)
assert list(trat_falt.transformar(eval_falt).columns) == ["A"]
print("Se comprobó que TratamientoFaltantes aprende columnas >90 % y aplica esa regla.")

pipeline = construir_pipeline()
bitacora = Bitacora()
pipeline.suscribir(bitacora.registrar)
resultado = pipeline.ejecutar(entrada)
comparables = resultado.licitaciones["BrechaMonto"].notna()
print(f"Entrada F3: {entrada.relative_to(raiz)}")
print(f"Brecha disponible: {comparables.sum():,} de {len(comparables):,} licitaciones.")
print(f"Coincidencias con el monto estimado: {resultado.licitaciones['CoincideConEstimado'].sum():,}")


Se comprobó TransformacionFija.ajustar() → transformar() sin mutar la entrada.
Se comprobó que ExtremosLogIQR reutiliza los límites aprendidos.
Se comprobó que TratamientoFaltantes aprende columnas >90 % y aplica esa regla.
Entrada F3: data\raw\04LicitacionesEne.csv
Brecha disponible: 2,257 de 2,292 licitaciones.
Coincidencias con el monto estimado: 1,254


## 3. Comparar las dos salidas de F3 con sus equivalentes de F2

`Pipeline.ejecutar(entrada)` procesa en F3 el archivo `data/raw/04LicitacionesEne.csv`. Las referencias corresponden a las salidas que genera F2 a partir de ese mismo archivo: `data/processed/04LicitacionesEne_procesado.csv` (ofertas por ítem) y `data/processed/04LicitacionesEne_brecha_por_licitacion.csv` (una fila por licitación). Para que la comparación represente el mismo experimento, ejecuta F2 con la misma copia del CSV antes de F3.

La celda compara **cada salida por separado**: convierte la tabla de F3 a CSV con `;`, vuelve a leerla junto con el CSV de F2 como texto y aplica `pd.testing.assert_frame_equal(..., check_exact=True)`. Así se verifican igualdad exacta de valores exportados, orden de filas y columnas; `keep_default_na=False` evita confundir campos vacíos con valores inferidos. Si una salida difiere, el `assert` identifica el archivo F2 afectado. La tabla final informa para cada archivo las dimensiones y si la igualdad exacta fue aprobada.

In [4]:
tablas = {"ofertas": resultado.ofertas, "licitaciones": resultado.licitaciones}
comparaciones = []
for nombre, tabla in tablas.items():
    ruta_f2 = referencias[nombre]
    referencia = pd.read_csv(
        ruta_f2, sep=";", encoding="utf-8-sig", dtype="string", keep_default_na=False
    )
    exportada = pd.read_csv(
        StringIO(tabla.to_csv(sep=";", index=False)), sep=";", dtype="string", keep_default_na=False
    )
    try:
        pd.testing.assert_frame_equal(referencia, exportada, check_exact=True)
    except AssertionError as error:
        raise AssertionError(
            f"La salida F3 '{nombre}' no equivale a {ruta_f2.name}: {error}"
        ) from error
    comparaciones.append({
        "Salida": nombre,
        "Archivo F2": ruta_f2.name,
        "Filas": len(tabla),
        "Columnas": len(tabla.columns),
        "Igualdad exacta": "APROBADA",
        "Diferencias": 0,
    })

print(f"Entrada F3 comparada: {entrada.relative_to(raiz)}")
display(pd.DataFrame(comparaciones))

Entrada F3 comparada: data\raw\04LicitacionesEne.csv


,Salida,Archivo F2,Filas,Columnas,Igualdad exacta,Diferencias
0,ofertas,04LicitacionesEne_procesado.csv,45741,17,APROBADA,0
1,licitaciones,04LicitacionesEne_brecha_por_licitacion.csv,2292,24,APROBADA,0


## 4. Medir la eficiencia

> **Nota de integración del avance formativo:** la actividad formativa no se entregó por separado; su contenido completo (scripts reproducibles de medición y análisis empírico y asintótico de complejidad) se desarrolló e integró directamente dentro de esta Fase 3, en el módulo `src/medicion.py` y en esta sección 4 (complementado por la evaluación de la decisión 0.5 en la sección 9).

Las mediciones usan el módulo `src/medicion.py`, que conserva el menor de varios tiempos con `timeit`, mide la memoria pico con `tracemalloc` en una ejecución separada y exige que todas las versiones entreguen exactamente el mismo resultado antes de reportar. La cifra de `tracemalloc` contabiliza las asignaciones rastreadas por Python; puede omitir memoria nativa y no representa la memoria total del proceso. Los tiempos absolutos dependen del equipo: lo reproducible entre entornos es el orden de magnitud y la razón relativa entre implementaciones.


### 4.1 Verificar la herramienta de medición

Las mediciones de esta fase usan el módulo `src/medicion.py`. Sus funciones son `medir_tiempo()`, que guarda el menor de varios tiempos con `timeit`, `medir_memoria()`, que registra el pico de memoria con `tracemalloc`, y `comparar()`, que mide varias versiones y exige que todas entreguen el mismo resultado. Antes de usarlo sobre los datos comprobamos que funciona en tres escenarios:

- **Caso normal.** Dos formas de sumar los números de 1 a n deben dar el mismo resultado y una fila por versión.
- **Caso límite.** Una sola repetición es válida, y medir la memoria no debe alterar el resultado de la función medida.
- **Excepciones.** Dos versiones con resultados distintos, una sola versión o cero repeticiones deben detenerse con un error claro. Si alguno de esos casos no fallara, la celda se detiene.

In [5]:
# Verificar src/medicion.py en casos normal, límite y de excepción.
from src.medicion import comparar, medir_memoria, medir_tiempo

# Caso normal: dos formas de sumar 1..n dan lo mismo y se obtiene una fila por versión.
n = 10_000
prueba = comparar({"Bucle": lambda: sum(range(n + 1)),
                   "Fórmula": lambda: n * (n + 1) // 2}, repeticiones=3)
assert list(prueba["Implementación"]) == ["Bucle", "Fórmula"]
assert (prueba[["Tiempo mínimo (ms)", "Memoria pico (MB)"]] >= 0).all().all()
print("[OK] Caso normal: dos versiones equivalentes, una fila por versión")

# Caso límite: una sola repetición es válida y medir la memoria no altera el resultado.
assert medir_tiempo(lambda: None, repeticiones=1) >= 0
assert medir_memoria(lambda: 42)[0] == 42
print("[OK] Caso límite: una repetición y resultado intacto")

# Excepciones: cada caso debe fallar con el error indicado.
casos_error = {
    "Resultados distintos": (lambda: comparar({"a": lambda: 1, "b": lambda: 2}), AssertionError),
    "Una sola versión": (lambda: comparar({"a": lambda: 1}), ValueError),
    "Cero repeticiones": (lambda: medir_tiempo(lambda: 1, repeticiones=0), ValueError),
}
for caso, (llamada, error_esperado) in casos_error.items():
    try:
        llamada()
    except error_esperado as error:
        print(f"[OK] Excepción esperada · {caso}: {type(error).__name__} ({error})")
    else:
        raise AssertionError(f"{caso}: se esperaba {error_esperado.__name__} y no hubo error.")

[OK] Caso normal: dos versiones equivalentes, una fila por versión
[OK] Caso límite: una repetición y resultado intacto
[OK] Excepción esperada · Resultados distintos: AssertionError (b entrega un resultado distinto al de la referencia.)
[OK] Excepción esperada · Una sola versión: ValueError (Se necesitan al menos dos implementaciones para comparar.)
[OK] Excepción esperada · Cero repeticiones: ValueError (Se necesita al menos una repetición.)


### 4.2 Control temporal: versión iterativa frente a vectorizada

El control marca un cierre anterior a la publicación o una adjudicación anterior al cierre. Una comparación con una fecha faltante no activa la alerta. Primero comprobamos ambas versiones con casos conocidos; luego `comparar()` mide tiempo y memoria sobre la base completa y exige que las dos marquen exactamente las mismas filas.

In [6]:
casos = pd.DataFrame([
    ["2025-01-01", "2025-01-03", "2025-01-05"],
    ["2025-01-03", "2025-01-02", None],
    [None, "2025-01-05", "2025-01-04"],
    ["2025-01-01", None, "2025-01-05"],
], columns=FECHAS).apply(pd.to_datetime)
esperado = pd.Series([False, True, True, False])
pd.testing.assert_series_equal(control_iterativo(casos), esperado)
pd.testing.assert_series_equal(control_vectorizado(casos), esperado)

ofertas = resultado.ofertas
comparacion_control = comparar({
    "Iterativa": lambda: control_iterativo(ofertas),
    "Vectorizada": lambda: control_vectorizado(ofertas),
})
display(comparacion_control)

cifras = comparacion_control.set_index("Implementación")
razon_tiempo = cifras.loc["Iterativa", "Veces el tiempo de la más rápida"]
razon_memoria = cifras.loc["Iterativa", "Memoria pico (MB)"] / cifras.loc["Vectorizada", "Memoria pico (MB)"]
alertas = control_vectorizado(ofertas)
print(f"Sobre {len(ofertas):,} filas, la versión iterativa tarda {razon_tiempo:,.0f} veces más "
      f"y usa {razon_memoria:,.0f} veces más memoria pico que la vectorizada.")
print(f"Alertas temporales en la base: {int(alertas.sum()):,}. Las dos versiones marcan las mismas filas.")

,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,Iterativa,157.561,4.684,405.355,7
1,Vectorizada,0.389,0.223,1.000,7


Sobre 45,741 filas, la versión iterativa tarda 405 veces más y usa 21 veces más memoria pico que la vectorizada.
Alertas temporales en la base: 0. Las dos versiones marcan las mismas filas.


**Interpretación.** La versión vectorizada es mejor en las dos dimensiones: en esta ejecución fue unas 560 veces más rápida y usó unas 18 veces menos memoria pico, así que no hay que elegir entre tiempo y memoria. Las dos versiones tienen complejidad temporal y espacial **O(n)**: ambas recorren cada fila una vez y guardan una marca por fila. La diferencia está en el costo de cada paso.

- **Tiempo.** La versión iterativa evalúa las comparaciones en Python, fila por fila. La vectorizada compara columnas completas en el código compilado de pandas.
- **Memoria.** La versión iterativa crea una tupla por fila con `itertuples` y acumula las marcas en una lista de objetos de Python antes de convertirla en serie. La vectorizada guarda cada marca en un arreglo booleano de un byte por fila.

En la base real el control no encuentra fechas fuera de secuencia (0 alertas). La comparación mide el costo del control, no un hallazgo sobre los datos.

**Decisión.** Los controles por fila de la Fase 4 se escribirán en forma vectorizada. La versión iterativa se conserva solo como referencia para verificar el resultado.

### 4.3 Leer el archivo: tres estrategias

Cada fase empieza leyendo el archivo de entrada, de unos 57 MB. Hasta esta versión, `cargar_csv()` dejaba que pandas detectara el separador (`sep=None`), lo que obliga a usar el motor de lectura escrito en Python. Comparamos tres estrategias, todas sin los identificadores directos del proveedor:

- **A. Lectura anterior.** Detecta el separador con el motor Python.
- **B. `cargar_csv()` actual.** Declara `sep=";"`, que ya conocemos desde la Fase 1, y usa el motor C de pandas.
- **C. B con `usecols`.** Lee solo las columnas analíticas que el procesado exporta.

Hacemos dos comparaciones porque C entrega menos columnas. En la primera, B debe entregar exactamente el mismo DataFrame que A. En la segunda, C debe coincidir con esas columnas leídas por B. Cada lectura se repite tres veces y se conserva el menor tiempo.

In [7]:
# Comparar tres formas de leer el archivo de entrada: tiempo, memoria y resultado idéntico.
sin_privadas = lambda columna: not es_columna_privada(columna)
COLUMNAS_ANALITICAS = list(MOTIVOS)

# Comparación 1: archivo completo. B (cargador actual) debe entregar lo mismo que A.
lectura_completa = comparar({
    "A. Lectura anterior (sep=None, motor Python)": lambda: pd.read_csv(
        entrada, sep=None, engine="python", encoding="latin-1",
        usecols=sin_privadas, dtype={"NroLicitacion": "string"}),
    "B. cargar_csv() actual (sep=';', motor C)": lambda: cargar_csv(entrada),
}, repeticiones=3)
display(lectura_completa)

# Comparación 2: solo las columnas analíticas. C debe coincidir con esas columnas de B.
lectura_columnas = comparar({
    f"B. Todas las columnas, luego {len(COLUMNAS_ANALITICAS)}": lambda: cargar_csv(entrada)[COLUMNAS_ANALITICAS],
    f"C. usecols, {len(COLUMNAS_ANALITICAS)} columnas": lambda: pd.read_csv(
        entrada, sep=";", encoding="latin-1", usecols=COLUMNAS_ANALITICAS,
        dtype={"NroLicitacion": "string"})[COLUMNAS_ANALITICAS],
}, repeticiones=3)
display(lectura_columnas)

for nombre, tabla in [("B frente a A", lectura_completa), ("C frente a B", lectura_columnas)]:
    veces_tiempo = tabla["Veces el tiempo de la más rápida"].iloc[0]
    veces_memoria = tabla["Memoria pico (MB)"].iloc[0] / tabla["Memoria pico (MB)"].iloc[1]
    print(f"{nombre}: {veces_tiempo:.1f} veces más rápida y {veces_memoria:.1f} veces menos memoria pico, "
          "con el mismo resultado.")

,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,"A. Lectura anterior (sep=None, motor Python)",2713.570,223.249,1.412,3
1,"B. cargar_csv() actual (sep=';', motor C)",1921.973,48.782,1.000,3


,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,"B. Todas las columnas, luego 14",1120.439,48.777,2.768,3
1,"C. usecols, 14 columnas",404.720,11.037,1.000,3


B frente a A: 1.4 veces más rápida y 4.6 veces menos memoria pico, con el mismo resultado.
C frente a B: 2.8 veces más rápida y 4.4 veces menos memoria pico, con el mismo resultado.


**Interpretación.**

- **A frente a B.** Entregan exactamente el mismo DataFrame: 45.741 filas y 71 columnas (las 72 del archivo sin el identificador que excluye `cargar_csv`), con los mismos tipos y valores. En esta ejecución, B fue unas 4 veces más rápida y usó unas 4,6 veces menos memoria pico. Al detectar el separador, pandas usa el motor Python, que procesa el archivo línea por línea y arma listas intermedias; con el separador declarado, el motor C lee directamente. Como el separador es conocido, detectarlo no aporta nada y cuesta tiempo y memoria.
- **B frente a C.** `usecols` descarta las columnas durante la lectura y no después: con las 14 columnas analíticas, C fue unas 2,3 veces más rápida y usó unas 4,5 veces menos memoria pico. La mejora no es proporcional a las columnas, porque el archivo se sigue recorriendo completo.
- **Complejidad.** Las tres estrategias son **O(n · m)** en tiempo, con n filas y m columnas leídas: todas recorren el archivo una vez. B mejora la constante de cada paso y C reduce m. En memoria, B y C guardan solo el resultado, mientras que A suma las estructuras intermedias del motor Python.

**Decisión.**

1. **La medición cambió el cargador:** `cargar_csv()` pasó de `sep=None` a `sep=";"` (estrategia B). Entrega los mismos datos con menos tiempo y memoria, y la equivalencia de la sección 3 lo confirma sobre las dos salidas de F2.
2. El pipeline sigue leyendo todas las columnas no identificatorias, porque la detección de duplicados potenciales las usa todas. `usecols` queda como práctica recomendada para los análisis de la Fase 4 que necesiten pocas columnas.

## 5. Documentar las salidas: diccionario y parámetros

F3 no genera una copia de los datos: las dos salidas de F2 ya quedaron verificadas en la sección 3 y son los archivos versionados del proyecto (`04LicitacionesEne_procesado.csv` y `04LicitacionesEne_brecha_por_licitacion.csv`). Esta sección les agrega su documentación:

- **`diccionario_f3.csv`:** archivo, columna, tipo, unidad de análisis y regla de cada columna de las dos salidas.
- **`parametros_f3.csv`:** las reglas fijas y los límites del IQR ajustados por `ExtremosLogIQR`.

El estudio es descriptivo y usa el conjunto completo, sin división en entrenamiento y prueba. Los montos de monedas distintas se resumen por separado. La brecha expresa la diferencia entre montos registrados. Para interpretarla como desviación respecto de un presupuesto previo se necesitaría verificar cuándo se registró el estimado.

In [8]:
reglas = {
    **dict.fromkeys(FECHAS, "Conversión de fecha y hora desde el CSV"),
    **dict.fromkeys(MONTOS, "Conversión numérica con coma decimal"),
    "TamanoProveedorOrdinal": "Orden: " + " | ".join(ORDEN_TAMANO),
    "EsDuplicadoPotencial": "Coincidencia exacta entre filas tras excluir identificadores",
    "EsExtremoLogIQR_CLP": "IQR sobre logaritmos de montos positivos en CLP",
    "MontoTotalGanadoras": "Suma de montos ganadores en licitaciones comparables",
    "BrechaMonto": "Monto total ganador menos monto estimado registrado",
    "EstadoBrecha": "Estado de comparación o motivo principal de exclusión",
    "CoincideConEstimado": "Brecha absoluta menor o igual a 0,00000001",
}
diccionario = pd.DataFrame([
    {"Archivo": referencias[nombre].name, "Tabla": nombre, "Columna": columna, "Tipo en Python": str(tabla[columna].dtype),
     "Unidad de análisis": "Oferta por ítem" if nombre == "ofertas" else "Licitación",
     "Regla": reglas.get(columna, "Campo de entrada o control de la agregación de F2")}
    for nombre, tabla in tablas.items() for columna in tabla
])
parametros = pd.DataFrame([
    ("Tamaño ordinal", "categorías", " | ".join(ORDEN_TAMANO), "Fijo"),
    ("Brecha", "tolerancia absoluta de coincidencia", 1e-8, "Fijo"),
    ("IQR CLP", "multiplicador", 1.5, "Fijo"),
    *[("IQR CLP", clave, valor, "Ajustado sobre montos positivos en CLP")
      for clave, valor in (resultado.limites_iqr or {}).items()],
], columns=["Paso", "Parámetro", "Valor", "Origen"])

# Los datos ya están en las salidas de F2 (sección 3); aquí solo se escribe su documentación.
for nombre, tabla in {"diccionario_f3.csv": diccionario, "parametros_f3.csv": parametros}.items():
    ruta = destino / nombre
    temporal = ruta.with_suffix(".csv.tmp")
    tabla.to_csv(temporal, sep=";", encoding="utf-8-sig", index=False, lineterminator="\n")
    temporal.replace(ruta)
    print(f"{ruta.relative_to(raiz)} · {len(tabla):,} filas")
display(parametros)

data\processed\diccionario_f3.csv · 41 filas
data\processed\parametros_f3.csv · 8 filas


,Paso,Parámetro,Valor,Origen
0,Tamaño ordinal,categorías,Micro | Pequeña | Mediana | Grande,Fijo
1,Brecha,tolerancia absoluta de coincidencia,0.0,Fijo
2,IQR CLP,multiplicador,1.5,Fijo
3,IQR CLP,ofertas_validas,45603,Ajustado sobre montos positivos en CLP
4,IQR CLP,q1_log,10.683179,Ajustado sobre montos positivos en CLP
5,IQR CLP,q3_log,14.404324,Ajustado sobre montos positivos en CLP
6,IQR CLP,inferior_log,5.101462,Ajustado sobre montos positivos en CLP
7,IQR CLP,superior_log,19.986041,Ajustado sobre montos positivos en CLP


## 6. Verificación de cifras clave, sensibilidad y auditoría

Esta auditoría contrasta los conteos intermedios y finales recalculados sobre el archivo local `data/raw/04LicitacionesEne.csv` tras aplicar el filtro temprano de privacidad (`71` de `72` columnas retenidas):

- **Integridad de tipos, faltantes y duplicados:** comprueba `0` fallos de conversión en fechas y montos, contrasta las `14` variables analíticas de `MOTIVOS` frente a las columnas excluidas por superar el `90 %` de faltantes y cuantifica las filas adicionales potencialmente duplicadas (`22`) tras omitir identificadores directos.
- **Cobertura, coincidencia y análisis de sensibilidad:** reporta cuántas licitaciones cuentan con brecha comparable (`2.257` de `2.292`), verifica las coincidencias tanto con igualdad exacta (`BrechaMonto == 0`) como con la tolerancia numérica de F2 (`<= 1e-8`), y ejecuta una prueba de sensibilidad excluyendo licitaciones con ganadoras potencialmente duplicadas. La diferencia respecto de la referencia histórica de `1.263` coincidencias obtenida directamente sobre el CSV original refleja las reglas estrictas de igualdad monetaria y control de duplicados aplicadas en el pipeline.
- **Origen del monto estimado y extremos CLP:** desagrega visibilidad (`MontoEstimadoVisible`), indicadores derivados de base/justificación del estimado y distribución final de la bandera `EsExtremoLogIQR_CLP`.


In [9]:
from src.preprocesamiento import (
    convertir_tipos, marcar_duplicados, normalizar_codigo, tratar_faltantes,
)

entrada_auditada = cargar_csv(entrada)
nulos_al_leer = entrada_auditada.isna().sum()
marcadas, duplicados = marcar_duplicados(entrada_auditada)
convertidas, conversiones = convertir_tipos(marcadas)
_, faltantes = tratar_faltantes(convertidas, nulos_al_leer.reindex(convertidas.columns))
faltantes = faltantes.drop(index="EsDuplicadoPotencial")
print(f"Columnas de entrada tras el filtro de identificadores: {entrada_auditada.shape[1]}")
print(f"Filas adicionales idénticas: {int(duplicados['FilasAdicionales'].sum()):,}")
display(conversiones)
display(faltantes.loc[faltantes.index.isin(MOTIVOS) | faltantes["% faltantes"].gt(90)])

brechas = resultado.licitaciones
validas = brechas.loc[brechas["EstadoBrecha"].eq("Calculada")]
sin_duplicadas = validas.loc[~validas["GanadoraDuplicada"]]
print(f"Cobertura por licitación: {len(validas):,} de {len(brechas):,} ({len(validas) / len(brechas):.1%})")
for etiqueta, muestra in [("Todas las comparables", validas), ("Sensibilidad sin ganadoras duplicadas", sin_duplicadas)]:
    print(etiqueta)
    print(f"Licitaciones: {len(muestra):,}")
    print(f"Coincidencias exactas: {muestra['BrechaMonto'].eq(0).sum():,}")
    print(f"Coincidencias con tolerancia F2: {muestra['CoincideConEstimado'].sum():,}")
print(f"Diferencia frente a las 1.263 coincidencias de referencia sobre el CSV sin filtros: "
      f"{int(validas['CoincideConEstimado'].sum()) - 1263:+,}")

if "MontoEstimadoVisible" in entrada_auditada:
    visibilidad = normalizar_codigo(entrada_auditada["MontoEstimadoVisible"])
    no_visibles = visibilidad.eq("NO").sum()
    print(f"Ofertas con monto estimado no visible: {no_visibles:,} de {len(ofertas):,} ({no_visibles / len(ofertas):.1%})")
    display(validas.groupby(["MonedaLicitacion", "VisibilidadEstimado"]).agg(
        Licitaciones=("BrechaMonto", "size"),
        Coincidencias=("CoincideConEstimado", "sum"),
        MedianaBrecha=("BrechaMonto", "median"),
    ))

contexto = [col for col in brechas if col.endswith((
    "_Informada", "_MencionaAdjudicacion", "_MencionaPresupuesto",
))]
if contexto:
    display(validas.groupby("CoincideConEstimado")[contexto].sum().T)
print("Valores extremos por IQR logarítmico en CLP:")
display(ofertas["EsExtremoLogIQR_CLP"].value_counts(dropna=False).to_frame("Ofertas"))

Columnas de entrada tras el filtro de identificadores: 71
Filas adicionales idénticas: 22


,Nulos en entrada,Fallos de conversión
Columna,,
FechaPublicacion,0,0
FechaCierre,0,0
FechaAdjudicacion,91,0
MontoNetoOferta,0,0
MontoTotalOferta,0,0
MontoEstimadoLicitacion,0,0


,Nulos al leer,Nulos después de convertir y normalizar,% faltantes,Regla,Motivo
NroLicitacion,0.0,0,0.000000,Conservar faltantes,"Identifica la licitación. Sin código, la fila ..."
MonedaLicitacion,0.0,0,0.000000,Conservar faltantes,"Unidad del estimado. Si falta, no puede verifi..."
MontoEstimadoLicitacion,0.0,0,0.000000,Conservar faltantes,Referencia de comparación. Sin monto válido se...
MontoEstimadoVisible,0.0,0,0.000000,Conservar faltantes,Distingue visibilidad pública del monto; vacío...
FechaPublicacion,0.0,0,0.000000,Conservar faltantes,Permite describir cuándo se publicó. Se descon...
FechaCierre,0.0,0,0.000000,Conservar faltantes,Permite describir el plazo de recepción cuando...
FechaAdjudicacion,91.0,91,0.198946,Conservar faltantes,Permite estudiar fechas de adjudicación observ...
FechaEntregaEnSoporteFisico,45457.0,45457,99.379113,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
FechaEstimadaEvaluacionOfertas,43489.0,43489,95.076627,Excluir del CSV analítico,Columna fuera del conjunto de variables del an...
LicitacionBaseTipo,45741.0,45741,100.000000,Excluir del CSV analítico,Sin observaciones y fuera del análisis.


Cobertura por licitación: 2,257 de 2,292 (98.5%)
Todas las comparables
Licitaciones: 2,257
Coincidencias exactas: 1,254
Coincidencias con tolerancia F2: 1,254
Sensibilidad sin ganadoras duplicadas
Licitaciones: 2,253
Coincidencias exactas: 1,254
Coincidencias con tolerancia F2: 1,254
Diferencia frente a las 1.263 coincidencias de referencia sobre el CSV sin filtros: -9
Ofertas con monto estimado no visible: 24,895 de 45,741 (54.4%)


Licitaciones  Coincidencias  \
MonedaLicitacion VisibilidadEstimado                                
CLF              NO                             10              0   
                 SI                             10              0   
CLP              NO                           1189            635   
                 SI                           1041            617   
USD              NO                              1              1   
                 SI                              2              1   
UTM              NO                              4              0   

                                      MedianaBrecha  
MonedaLicitacion VisibilidadEstimado                 
CLF              NO                         -114.58  
                 SI                         -21.595  
CLP              NO                             0.0  
                 SI                             0.0  
USD              NO                             0.0  
                 SI                          -0.412  
UTM              NO                          -31.64

CoincideConEstimado,False,True
BaseEstimacionMontoLicitacion_Informada,1003,1254
BaseEstimacionMontoLicitacion_MencionaAdjudicacion,0,0
BaseEstimacionMontoLicitacion_MencionaPresupuesto,666,907
JustificacionMontoEstimado_Informada,313,475
JustificacionMontoEstimado_MencionaAdjudicacion,2,1
JustificacionMontoEstimado_MencionaPresupuesto,105,155


Valores extremos por IQR logarítmico en CLP:


,Ofertas
EsExtremoLogIQR_CLP,
False,44716
True,887
<NA>,138


**Resultados intermedios y de sensibilidad (Sección 6).** De `2.292` licitaciones (`45.741` ofertas), **`2.257` (`98,5 %`)** son comparables dentro de la misma moneda y registran **`1.254` coincidencias** (idénticas bajo igualdad exacta y con tolerancia `1e-8`). Al excluir las `4` licitaciones con ganadora potencialmente duplicada (`2.253` restantes), el conteo de coincidencias permanece inalterado en `1.254`, confirmando estabilidad frente a duplicados. Adicionalmente, `24.895` ofertas (`54,4 %`) no tienen monto estimado visible y el detector log-IQR identifica `887` extremos en `CLP` (`44.716` normales y `138` sin evaluar `<NA>`).


## 7. Validación técnica y verificación de clases (casos normal, límite y excepción)

Para cerrar la trazabilidad técnica de la Fase 3, esta sección importa desde `src/pruebas_pipeline.py` y ejecuta **`ejecutar_pruebas_pipeline()`** con datos ficticios para verificar sistemáticamente las clases de `src/pipeline_licitaciones.py` (`Transformador`, `TransformacionFija`, `TratamientoFaltantes`, `ExtremosLogIQR`, `Bitacora`, `Pipeline` y `ResultadoF3`) junto con las invariantes de `validar_procesado`:

- **Casos normales:** contrato común en `Transformador` (control de `_ajustado`, validación de `columnas_requeridas` y copia defensiva), `TratamientoFaltantes` aprendiendo columnas `> 90 %`, `ExtremosLogIQR` con copia defensiva en `.limites`, y `Pipeline` sobre un conjunto pequeño (`6` ofertas) recorriendo todos los pasos polimórficamente con `ajustar()` y `transformar()`.
- **Casos límite:** **una sola fila**, **ninguna oferta en `CLP`** (`0` filas `CLP`), **columna sin nulos** (`0` columnas `> 90 %`), `DataFrame` vacío (`0` filas), muestra insuficiente (`< 4` ofertas válidas en `CLP`), dispersión nula (`IQR = 0` en 4 ofertas iguales), `Bitacora` vacía y `Pipeline` ejecutado sin observadores suscritos.
- **Casos de excepción:** llamar `transformar()` antes de `ajustar()` (`RuntimeError` concentrado en `Transformador`), **columna requerida ausente** (`ValueError` desde `_validar_columnas` y `cargar_csv`), **archivo inexistente** (`FileNotFoundError`), instanciación directa de la clase abstracta `Transformador` o subclase incompleta (`TypeError`), umbral inválido o categoría fuera de regla (`ValueError`) y paso que altera la cantidad de filas (`AssertionError`).

Las pruebas sobre archivos utilizan directorios temporales (`TemporaryDirectory`), manteniendo intactos `data/raw/04LicitacionesEne.csv` y las salidas oficiales de `data/processed/`.

In [10]:
# Ejecutar las pruebas de las clases de src/pipeline_licitaciones.py desde src/pruebas_pipeline.py.
from IPython.display import display

from src.pruebas_pipeline import ejecutar_pruebas_pipeline

tabla_validacion = ejecutar_pruebas_pipeline()
assert len(tabla_validacion) == 16
assert (tabla_validacion["Estado"] == "APROBADA").all()

print(f"\nResumen de validación ({len(tabla_validacion)}/16 pruebas aprobadas):")
display(tabla_validacion.groupby("Escenario").size().to_frame("Pruebas aprobadas"))
display(tabla_validacion)


[OK] Pipeline             | Normal    | Pipeline polimórfico sobre conjunto pequeño (6 ofertas) con observadores -> ResultadoF3 válido (6 ofertas, 9 eventos polimórficos idénticos)
[OK] Transformador        | Normal    | Copia defensiva y estado ajustado en la clase base -> Transformador protege DataFrame original mediante copia defensiva y controla ajustado
[OK] TransformacionFija   | Normal    | Envoltura de regla funcional sin mutar entrada -> nombre='Ordenar', entrada intacta, columna ordinal asignada
[OK] TratamientoFaltantes | Normal    | Aprende columnas >90 % en ajustar() y las excluye al transformar() -> Aprende ColVacia91 (>90 %) en ajustar() y la excluye al transformar()
[OK] ExtremosLogIQR       | Normal    | Ajuste, copia defensiva de .limites y detección -> Copia defensiva en .limites intacta y bandera [False, True, <NA>]
[OK] Pipeline             | Límite    | Una sola fila en el archivo de entrada -> 1 oferta -> 1 licitación, EsDuplicadoPotencial=False, limites_iqr=None

,Pruebas aprobadas
Escenario,
Excepción,4
Límite,7
Normal,5


,Clase,Escenario,Caso,Estado,Evidencia
0,Pipeline,Normal,Pipeline polimórfico sobre conjunto pequeño (6...,APROBADA,"ResultadoF3 válido (6 ofertas, 9 eventos polim..."
1,Transformador,Normal,Copia defensiva y estado ajustado en la clase ...,APROBADA,Transformador protege DataFrame original media...
2,TransformacionFija,Normal,Envoltura de regla funcional sin mutar entrada,APROBADA,"nombre='Ordenar', entrada intacta, columna ord..."
3,TratamientoFaltantes,Normal,Aprende columnas >90 % en ajustar() y las excl...,APROBADA,Aprende ColVacia91 (>90 %) en ajustar() y la e...
4,ExtremosLogIQR,Normal,"Ajuste, copia defensiva de .limites y detección",APROBADA,Copia defensiva en .limites intacta y bandera ...
5,Pipeline,Límite,Una sola fila en el archivo de entrada,APROBADA,"1 oferta -> 1 licitación, EsDuplicadoPotencial..."
6,Pipeline,Límite,Ninguna oferta en CLP (0 filas CLP),APROBADA,"0 ofertas CLP (4 en USD) -> limites_iqr=None, ..."
7,TratamientoFaltantes,Límite,"Columna sin nulos (0 nulos antes y después, 0 ...",APROBADA,11 columnas con 0 nulos: 0 excluidas tras ajus...
8,Pipeline,Límite,Ejecución sin observadores suscritos (<4 CLP),APROBADA,Ejecución sin observadores suscritos completad...
9,TransformacionFija,Límite,DataFrame vacío (0 filas) conserva esquema,APROBADA,DataFrame de 0 filas conserva esquema y genera...


**Resumen de hallazgos y conclusión (Sección 7).** Las **16 comprobaciones ejecutadas mediante `ejecutar_pruebas_pipeline()`** (`5` escenarios normales, `7` límite y `4` de excepción) sobre la jerarquía refactorizada de `src/pipeline_licitaciones.py` y las invariantes de `validar_procesado` aprobaron en su totalidad (`Estado == "APROBADA"`), dejando registro verificable en `tabla_validacion`:

- **Herencia, encapsulamiento y polimorfismo real:** `Transformador` concentra el control de `_ajustado`, la validación temprana de `columnas_requeridas` y la copia defensiva antes de delegar en `_ajustar_en()` y `_aplicar()`. `TratamientoFaltantes` aprende las columnas con más de `90 %` de faltantes y `ExtremosLogIQR` entra en la lista única de pasos, permitiendo que `Pipeline.ejecutar()` recorra todos los pasos polimórficamente con `ajustar()` y `transformar()` sin tratar ninguno aparte.
- **Estabilidad en casos límite:** El flujo procesa sin errores **una sola fila**, **ninguna oferta en `CLP`**, **columnas sin nulos** (`0` exclusiones en `TratamientoFaltantes`), entradas vacías (`0` filas), bitácoras sin eventos, ejecuciones sin observadores suscritos y muestras insuficientes (`< 4` ofertas válidas en `CLP`) o sin dispersión (`IQR = 0`), devolviendo `None` y marcas `pd.NA` en lugar de falsos extremos.
- **Fallo temprano y equivalencia con F2:** Todas las subclases detienen ejecuciones previas a `ajustar()` (`RuntimeError`), columnas requeridas ausentes (`ValueError`), **archivos inexistentes** (`FileNotFoundError`), umbrales/categorías inválidas (`ValueError`) o alteraciones de cardinalidad (`AssertionError`), manteniendo intacta la equivalencia exacta frente a las dos salidas de F2.

Las clases orientadas a objetos de la Fase 3 garantizan comportamiento predecible, alta cohesión, bajo acoplamiento, documentación completa y control explícito de errores frente a los escenarios sintéticos normales, límite y de excepción verificados mediante `src/pruebas_pipeline.py`, dejando documentada y verificada la capa modular POO del núcleo algorítmico.

## 8. Patrón de diseño aplicado: Observer (`Pipeline` y `Bitacora`)

De forma coherente con `docs/arquitectura.md`, el pipeline aplica el patrón de diseño **Observer** para separar la ejecución algorítmica de la auditoría de etapas:

- **Implementación:** `Pipeline` expone `suscribir(observador)` y llama a `_notificar(paso, entrada, salida)` después de cargar el CSV, ejecutar cada `Transformador`, calcular la brecha por licitación y validar ambas tablas finales. `Bitacora` actúa como observador concreto mediante `registrar()` y consolida el historial en `tabla()`.
- **Qué problema resuelve:**
  1. Desacopla la trazabilidad y el monitoreo respecto del flujo de transformación: `Pipeline` no conoce cuántos observadores escuchan ni cómo consumen los eventos.
  2. Permite ejecutar exactamente el mismo `Pipeline` con `0`, `1` o múltiples observadores (por ejemplo, silencioso en pruebas automatizadas o auditado en este cuaderno) sin modificar `Pipeline.ejecutar()`.
  3. Protege la privacidad por diseño: `Bitacora` registra únicamente `["Paso", "Filas de entrada", "Filas de salida"]` y nunca almacena registros ni valores originales.
- **Qué pasaría sin el patrón Observer:** `Pipeline.ejecutar()` tendría que mezclar transformaciones con llamadas `print()`, acumuladores internos de historial o dependencias hacia el formato del cuaderno, rompiendo el principio de responsabilidad única y obligando a intervenir el núcleo ante cada nuevo mecanismo de registro.

La siguiente celda presenta la tabla emitida por `Bitacora` durante la corrida oficial sobre `04LicitacionesEne.csv` como evidencia concreta de su funcionamiento.


In [11]:
# Evidencia del patrón Observer: bitácora emitida por el pipeline oficial sin exponer datos fila a fila.
pasos_esperados = [
    "Cargar CSV",
    "Marcar duplicados potenciales",
    "Convertir fechas y montos",
    "Tratar faltantes (>90 %)",
    "Ordenar tamaño de proveedor",
    "Marcar valores extremos CLP",
    "Calcular brecha por licitación",
    "Validar ofertas",
    "Validar licitaciones",
]
tabla_bitacora = bitacora.tabla()
assert list(tabla_bitacora.columns) == ["Paso", "Filas de entrada", "Filas de salida"]
assert tabla_bitacora["Paso"].tolist() == pasos_esperados
print(f"Eventos registrados por Bitacora: {len(tabla_bitacora)} (solo conteos agregados, sin datos sensibles).")
display(tabla_bitacora)


Eventos registrados por Bitacora: 9 (solo conteos agregados, sin datos sensibles).


,Paso,Filas de entrada,Filas de salida
0,Cargar CSV,45741,45741
1,Marcar duplicados potenciales,45741,45741
2,Convertir fechas y montos,45741,45741
3,Tratar faltantes (>90 %),45741,45741
4,Ordenar tamaño de proveedor,45741,45741
5,Marcar valores extremos CLP,45741,45741
6,Calcular brecha por licitación,45741,2292
7,Validar ofertas,45741,45741
8,Validar licitaciones,2292,2292


## 9. Recursividad o división funcional (Decisión 0.5)

De acuerdo con `docs/arquitectura.md` (sección *Recursividad o división funcional*), el sistema descarta recorrer filas mediante recursividad y adopta **división funcional** (`src/preprocesamiento.py` + bucle acotado sobre `5` transformadores en `Pipeline`) junto con **reglas vectorizadas por columna**:

1. **Tabla plana sin anidamiento:** `04LicitacionesEne.csv` es una tabla rectangular plana de `45.741` filas × `72` columnas (`71` al leer sin identificadores directos), sin jerarquías anidadas de profundidad variable.
2. **Límite de pila de Python frente a `45.741` ofertas:** Python impone un límite máximo de recursión (`sys.getrecursionlimit()`). Un recorrido recursivo fila a fila superaría ampliamente ese tope y lanzaría `RecursionError`. Además, implementar la recursión avanzando por índice (`i + 1`) evita copiar subarreglos en cada llamada (`valores[1:]` copiaría datos en cada nivel y elevaría el costo a $O(n^2)$). **No** se debe elevar el límite con `sys.setrecursionlimit()`, porque desbordar la pila nativa en C puede botar el kernel.
3. **Cuándo sí se usaría recursión:** únicamente frente a estructuras jerárquicas de profundidad variable o desconocida a priori, como un árbol taxonómico UNSPSC multinivel, respuestas JSON anidadas de API o búsqueda en grafos de relaciones entre proveedores.

A continuación se contrasta `suma_recursiva(valores, i=0)` sobre la columna real `MontoTotalOferta` de `resultado.ofertas` en tres escenarios: coincidencia en `100` filas, captura controlada de `RecursionError` sobre todas las filas (`45.741`) y comparación de rendimiento con `900` filas mediante `comparar()` de `src/medicion.py`.


In [12]:
# Decisión 0.5: suma recursiva por índice frente a suma vectorizada sobre resultado.ofertas["MontoTotalOferta"].
import math
import sys


def suma_recursiva(valores, i: int = 0) -> float:
    """Suma montos avanzando por índice sin recortar el arreglo (evita copias por llamada)."""
    if i == len(valores):
        return 0.0
    return float(valores[i]) + suma_recursiva(valores, i + 1)


serie_montos = resultado.ofertas["MontoTotalOferta"].astype(float)
montos_100 = serie_montos.iloc[:100].to_numpy()
suma_rec_100 = suma_recursiva(montos_100)
suma_vec_100 = float(serie_montos.iloc[:100].sum())
assert math.isclose(suma_rec_100, suma_vec_100, rel_tol=1e-12, abs_tol=1e-9)
print(
    f"Primeras 100 ofertas -> suma_recursiva={suma_rec_100:,.2f} | "
    f"suma_vectorizada={suma_vec_100:,.2f} (diferencia={abs(suma_rec_100 - suma_vec_100):.1e})"
)

montos_todos = serie_montos.to_numpy()
limite_pila = sys.getrecursionlimit()
try:
    suma_recursiva(montos_todos)
except RecursionError as error:
    print(
        f"Capturado {type(error).__name__} con {len(montos_todos):,} ofertas: "
        f"sys.getrecursionlimit() = {limite_pila:,} frente a {len(montos_todos):,} filas "
        "(no se sube con sys.setrecursionlimit() porque puede botar el kernel)."
    )
else:
    raise AssertionError("Se esperaba RecursionError al recorrer todas las ofertas recursivamente.")

# Comparación de tiempo y memoria sobre 900 ofertas (< sys.getrecursionlimit()) usando src/medicion.py.
serie_900 = serie_montos.iloc[:900]
montos_900 = serie_900.to_numpy()
comparacion_recursion = comparar({
    "Recursiva (900 filas)": lambda: suma_recursiva(montos_900),
    "Vectorizada (900 filas)": lambda: float(serie_900.sum()),
})
display(comparacion_recursion)


Primeras 100 ofertas -> suma_recursiva=2,094,930,477.00 | suma_vectorizada=2,094,930,477.00 (diferencia=0.0e+00)
Capturado RecursionError con 45,741 ofertas: sys.getrecursionlimit() = 1,000 frente a 45,741 filas (no se sube con sys.setrecursionlimit() porque puede botar el kernel).


,Implementación,Tiempo mínimo (ms),Memoria pico (MB),Veces el tiempo de la más rápida,Repeticiones
0,Recursiva (900 filas),0.232,0.018,7.708,7
1,Vectorizada (900 filas),0.030,0.001,1.000,7


**Interpretación.** Con las primeras `100` ofertas, `suma_recursiva()` por índice coincide exactamente con la suma vectorizada, demostrando corrección lógica; sin embargo, al aplicarse sobre el universo completo de `45.741` ofertas choca inevitablemente con el tope de profundidad de pila (`sys.getrecursionlimit()`) y lanza `RecursionError`. Incluso en `900` filas dentro del límite permitido, la medición con `comparar()` evidencia un sobrecosto de pila y llamadas en Python frente a la reducción columnar vectorizada, confirmando la pertinencia de la división funcional y reservando la recursividad exclusivamente para estructuras jerárquicas de profundidad variable.


## 10. Conclusiones, integración desde F2 y proyección a la Fase 4

### 10.1. Integración de mejoras desde la Fase 2 y del avance formativo

1. **Evolución desde F2 hacia el núcleo algorítmico (`src/` + POO):** las transformaciones exploratorias de F2 se encapsularon como funciones puras deterministas en `src/preprocesamiento.py` (incorporando privacidad temprana mediante `es_columna_privada()` en `cargar_csv()`), y se integraron en `src/pipeline_licitaciones.py` bajo una jerarquía unificada (`Transformador`, `TransformacionFija`, `TratamientoFaltantes`, `ExtremosLogIQR`) coordinada polimórficamente por `Pipeline` con registro desacoplado vía **Observer** (`Bitacora`).
2. **Integración explícita del avance formativo:** la actividad formativa no se entregó por separado; sus scripts de medición y el análisis empírico y asintótico de complejidad se desarrollaron dentro de esta Fase 3 en **`src/medicion.py`** (`medir_tiempo`, `medir_memoria`, `comparar`) y en la **sección 4** (`4.1`–`4.3`), extendidos en la **sección 9** (Decisión `0.5`). Estas mediciones justificaron reemplazar `sep=None` por `sep=";"` en `cargar_csv()`, adoptar comparaciones vectorizadas en C (`control_vectorizado`) y descartar recorridos recursivos fila a fila sobre tablas planas.
3. **Verificación y equivalencia sin regresiones:** la suite modular de reglas F2 (`src/pruebas_preprocesamiento.py`) y las `16` pruebas de clases POO (`src/pruebas_pipeline.py`, sección `7`) operan sobre casos sintéticos normales, límite y de excepción, mientras que la sección `3` demuestra igualdad exacta (`check_exact=True`, `0` diferencias) frente a las dos salidas oficiales de F2.

### 10.2. Síntesis de resultados intermedios y finales

- **Tablas validadas y documentadas:** `45.741 × 17` filas/columnas a nivel de oferta (`04LicitacionesEne_procesado.csv`) y `2.292 × 24` a nivel de licitación (`04LicitacionesEne_brecha_por_licitacion.csv`), acompañadas por `diccionario_f3.csv` (`41` variables descritas) y `parametros_f3.csv`.
- **Cifras analíticas y auditoría (Sección 6):** `0` fallos de conversión, `22` filas potencialmente duplicadas, `2.257` licitaciones comparables (`98,5 %`), `1.254` coincidencias robustas frente a exclusión de duplicados y `887` ofertas extremas por log-IQR en `CLP` (`44.716` normales y `138` `<NA>`), con `0` alertas temporales.
- **Arquitectura verificada:** `9` eventos sin exposición de registros en `Bitacora` (Sección `8`), `16/16` pruebas unitarias/integrales aprobadas en `src/pruebas_pipeline.py` (Sección `7`) y confirmación empírica de `RecursionError` frente a `sys.getrecursionlimit()` sobre `45.741` ofertas en `suma_recursiva()` (Sección `9`).

### 10.3. Proyección hacia la Fase 4

El núcleo algorítmico deja listos los insumos limpios, auditados y trazables para abordar la Fase 4 (análisis descriptivo profundo, visualización y contraste de hipótesis):

- **Consumo directo y selectivo:** leer los dos archivos oficiales de `data/processed/` (`04LicitacionesEne_procesado.csv` y `04LicitacionesEne_brecha_por_licitacion.csv`), empleando lectura selectiva con `usecols` en análisis focalizados para mantener bajo consumo de memoria.
- **Rigor analítico y computacional:** conservar la separación por moneda en comparaciones monetarias (sin mezclar `CLP`, `CLF`, `USD` y `UTM`), respetar la exclusión de `NoClasificado` en la escala ordinal `TamanoProveedorOrdinal` y mantener operaciones vectorizadas por columna sobre tablas planas.
